In [11]:
# Paso 1 · Importar librerías

# pandas: maneja tablas de datos como un excle pero con codigo
import pandas as pd

# matplotlib: hace gráficos
import matplotlib.pyplot as plt

# Calendario de feriados de EE. UU., que trae pandas incluido (Se usa debido al dataset de ese país)
from pandas.tseries.holiday import USFederalHolidayCalendar


In [12]:
# Paso 2 · Cargar y mirar los datos
# Qué hacemos: leer el CSV y mirar su forma antes de tocar nada.
# Por qué: no podés limpiar algo que no conocés. Este paso responde: cuántos datos hay, qué columnas, de qué tipo.

# Leemos el CSV. La ruta empieza con ".." porque el notebook está dentro de "notebooks/"
# y tiene que subir una carpeta para llegar a "data/".
# decimal="," le indica que en este archivo los decimales vienen con coma
# (formato argentino), no con punto como en el original
df = pd.read_csv("../data/raw/seattle_tons_trips.csv", decimal=",")

#shape devuelve (cantidad de filas, cantidad de columnas)
print(df.shape)

#dtypes muestra el tipoi de dato de cada columna
print (df.dtypes)

#head() muestra las primeras 5 filas (es la última línea, asique se muestra sola)
df.head()

(36215, 8)
YEAR                  int64
MONTH                 int64
Date                    str
Transfer Station        str
COMMODITY               str
TONS                float64
TRIPS                 int64
SECTOR                  str
dtype: object


,YEAR,MONTH,Date,Transfer Station,COMMODITY,TONS,TRIPS,SECTOR
0,2009,5,05/04/2009,North Transfer Station,Garbage,131.514968,23,Residential
1,2009,5,05/04/2009,South Transfer Station,Recycling,315.151139,70,Residential
2,2009,5,05/05/2009,South Transfer Station,Garbage,69.426921,15,Residential
3,2009,5,05/05/2009,South Transfer Station,Recycling,317.744168,71,Residential
4,2009,5,05/05/2009,South Transfer Station,Organics,198.330000,33,Residential


In [13]:
# Paso 3 · Renombrar columnas y convertir la fecha
# Qué hacemos: pasar los nombres al castellano, convertir la fecha a tipo fecha
# y traducir los valores de estación y tipo de residuo.
# Por qué: nombres en castellano hacen el código más fácil de leer;
# como texto, la fecha no se puede ordenar, filtrar por año ni saber qué día de la semana fue.

# rename recibe un diccionario {nombre_viejo: nombre_nuevo}
df = df.rename(columns={
    "Date": "fecha",
    "Transfer Station": "estacion",
    "COMMODITY": "tipo",
    "TONS": "toneladas",
    "TRIPS": "viajes",
    "SECTOR": "sector",
})

# Convertimos el texto a fecha. %m = mes, %d = día, %Y = año con 4 dígitos.
# El format evita que pandas confunda el 5 de abril con el 4 de mayo,
# porque el dataset usa el formato estadounidense mes/día/año
df["fecha"] = pd.to_datetime(df["fecha"], format="%m/%d/%Y")

# map reemplaza cada valor según el diccionario
df["estacion"] = df["estacion"].map({
    "North Transfer Station": "Norte",
    "South Transfer Station": "Sur",
})
df["tipo"] = df["tipo"].map({
    "Garbage": "Basura",
    "Recycling": "Reciclables",
    "Organics": "Organicos",
})

# Verificamos: ninguna estación o tipo tiene que haber quedado vacío.
# Si map encuentra un valor que no está en el diccionario, lo deja vacío (NaN)
print("Vacíos en estación:", df["estacion"].isna().sum())
print("Vacíos en tipo:", df["tipo"].isna().sum())

df.dtypes

Vacíos en estación: 0
Vacíos en tipo: 0


YEAR                  int64
MONTH                 int64
fecha        datetime64[us]
estacion                str
tipo                    str
toneladas           float64
viajes                int64
sector                  str
dtype: object

In [ ]:
# Paso 4 · Valores faltantes y duplicados exactos

# Qué hacemos: contar celdas vacías y filas idénticas.

# Por qué: la consigna pide revisarlo y explicar las decisiones. Un vacío puede hacer fallar el modelo, y una fila copiada cuenta dos veces el mismo dato.

# isna() marca True donde hay un vacío; sum() cuenta cuántos True hay por columna
print("Faltantes por columna:")
print(df.isna().sum())

# duplicated() marca True en las filas idénticas a una anterior, en TODAS las columnas
print("\nFilas duplicadas exactas:", df.duplicated().sum())

Faltantes por columna:
YEAR         0
MONTH        0
fecha        0
estacion     0
tipo         0
toneladas    0
viajes       0
sector       0
dtype: int64

Filas duplicadas exactas: 0


In [ ]:
# No se encontraron valores faltantes ni filas duplicadas exactas, por lo que no fue necesario eliminar ni completar datos en este paso.

In [15]:
# Paso 5 · Columnas que no aportan

# Qué hacemos: contar cuántos valores distintos tiene cada columna y eliminar las que no sirven.

# Por qué: una columna con un único valor es igual para todos los casos, así que no ayuda a distinguir nada. Y las columnas que repiten información de otra solo agregan riesgo de contradicciones.

# nunique() cuenta los valores distintos de cada columna
print(df.nunique())

# Eliminamos:
# - sector: tiene un solo valor ("Residential"), no distingue ningún caso
# - YEAR y MONTH: son información que ya está en la fecha;
#   en el Paso 11 las volvemos a crear a partir de ella, así hay una sola fuente
df = df.drop(columns=["sector", "YEAR", "MONTH"])

print("\nColumnas que quedan:", df.columns.tolist())

YEAR            18
MONTH           12
fecha         6386
estacion         2
tipo             3
toneladas    34540
viajes         126
sector           1
dtype: int64

Columnas que quedan: ['fecha', 'estacion', 'tipo', 'toneladas', 'viajes']


In [ ]:
#Se eliminó sector por tener un único valor (todos los registros son residenciales). Se eliminaron YEAR y MONTH porque se obtienen de la fecha; mantener una sola fuente evita inconsistencias